# Using eBFE Models: Spring River Validation

This notebook validates the organized Spring River eBFE delivery as a
results-ready HEC-RAS 6.1 example. It uses the shared delivery workspace,
reviews the saved audit evidence, executes a fresh geometry-preprocessor
validation run, confirms the local compatibility assets that distinguish
Spring River from Spring Creek, and verifies that all seven plan result
HDFs resolve inside the organized RAS project folder.


## What This Notebook Proves

1. `RasEbfeModels.organize_model("spring-river")` resolves to the shared organized delivery for HUC 11010010.
2. The organized study remains clearly distinct from `spring-creek` / `SpringCreek_12040102` in code, folder names, and notebook messaging.
3. The organized RAS project has local projection, terrain, `Land Cover`, legacy `Land Classification`, DSS, and RASMapper assets.
4. The notebook validation run executes the HEC-RAS 6.1 geometry preprocessor for plan 01 / geometry 01 and archives fresh review evidence under the issue artifact root.
5. All seven hydraulic result HDFs are present and resolve from `ras.plan_df` inside the organized project folder.


In [1]:
from pathlib import Path
import json
import logging
import os
import sys

import h5py
import pandas as pd
from IPython.display import display

def find_repo_root(start: Path | None = None) -> Path:
    search_start = (start or Path.cwd()).resolve()
    for candidate in [search_start, *search_start.parents]:
        if (candidate / "scripts" / "ebfe_geometry_preprocessor_batch.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root from current working directory.")

try:
    from ras_commander import RasPrj, RasUtils, init_ras_project
    from ras_commander.sources.federal import RasEbfeModels
except ImportError:
    repo_root = find_repo_root()
    sys.path.insert(0, str(repo_root))
    from ras_commander import RasPrj, RasUtils, init_ras_project
    from ras_commander.sources.federal import RasEbfeModels

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("ras_commander").setLevel(logging.WARNING)
pd.options.display.max_colwidth = 160


## Configure the eBFE Workspace


In [2]:
EBFE_WORKSPACE = Path(os.environ.get("RAS_COMMANDER_EBFE_ROOT", "H:/Testing/eBFE Model Organization"))
DOWNLOAD_ROOT = EBFE_WORKSPACE / "Downloads"
ORGANIZED_ROOT = EBFE_WORKSPACE / "Organized"
VALIDATION_ROOT = EBFE_WORKSPACE / "Validation" / "ebfe_delivery"
REVIEW_ROOT = Path(os.environ.get("RAS_COMMANDER_REVIEW_ROOT", "H:/Symphony/ras-commander/CLB-215"))
NOTEBOOK_ARTIFACT_ROOT = REVIEW_ROOT / "spring_river_notebook"
NOTEBOOK_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

organized_folder = ORGANIZED_ROOT / "SpringRiver_11010010"
if not organized_folder.exists():
    organized_folder = RasEbfeModels.organize_model(
        "spring-river",
        download_root=DOWNLOAD_ROOT,
        output_root=ORGANIZED_ROOT,
    )

ras_model_root = organized_folder / "RAS Model"
print(f"Spring River organized folder: {organized_folder}")
print(f"RAS Model root exists: {ras_model_root.exists()}")
print(f"Validation root: {VALIDATION_ROOT}")
print(f"Review artifact root: {NOTEBOOK_ARTIFACT_ROOT}")


Spring River organized folder: H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010
RAS Model root exists: True
Validation root: H:\Testing\eBFE Model Organization\Validation\ebfe_delivery
Review artifact root: H:\Symphony\ras-commander\CLB-215\spring_river_notebook


## Confirm Model Identity and Naming


In [3]:
projects = RasUtils.find_valid_ras_folders(
    ras_model_root,
    max_depth=10,
    return_project_info=True,
)
project_df = pd.DataFrame(projects)
project_folder = Path(project_df.iloc[0]["folder"])

model_identity = pd.DataFrame(
    [
        {
            "model_key": "spring-river",
            "huc8": "11010010",
            "organized_folder": organized_folder.name,
            "project_name": project_df.iloc[0]["project_name"],
            "ras_version": "6.1",
            "distinguishes_from": "spring-creek / SpringCreek_12040102",
        }
    ]
)
model_identity


,model_key,huc8,organized_folder,project_name,ras_version,distinguishes_from
0,spring-river,11010010,SpringRiver_11010010,Spring_BLE,6.1,spring-creek / SpringCreek_12040102


## Discover and Initialize the RAS Project


In [4]:
display(project_df[["project_name", "folder", "prj_file"]])

ras = RasPrj()
init_ras_project(
    project_folder,
    "6.1",
    ras_object=ras,
    load_results_summary=False,
)

plan_columns = [
    column for column in ["plan_number", "flow_type", "Geom File", "Flow File", "HDF_Results_Path"]
    if column in ras.plan_df.columns
]
display(ras.plan_df[plan_columns].sort_values("plan_number"))


,project_name,folder,prj_file
0,Spring_BLE,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.prj


,plan_number,flow_type,Geom File,Flow File,HDF_Results_Path
2,01,Unsteady,01,02,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p01.hdf
0,02,Unsteady,01,01,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p02.hdf
1,03,Unsteady,01,04,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p03.hdf
3,04,Unsteady,01,03,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p04.hdf
4,05,Unsteady,01,05,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p05.hdf
5,06,Unsteady,01,06,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p06.hdf
6,07,Unsteady,01,07,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p07.hdf


## Confirm Local Delivery Assets and Compatibility Copies


In [5]:
asset_rows = []
for label, pattern in {
    "projection": "Projection/*.prj",
    "terrain_hdf": "Terrain/**/*.hdf",
    "land_cover_tif": "Land Cover/**/*.tif",
    "land_cover_hdf": "Land Cover/**/*.hdf",
    "legacy_land_classification_tif": "Land Classification/**/*.tif",
    "legacy_land_classification_hdf": "Land Classification/**/*.hdf",
    "dss": "DSS Inputs/*.dss",
    "rasmap": "*.rasmap",
}.items():
    for path in sorted(project_folder.glob(pattern)):
        asset_rows.append(
            {
                "asset_type": label,
                "file": path.name,
                "relative_path": str(path.relative_to(project_folder)),
                "size_mb": round(path.stat().st_size / 1_000_000, 3),
            }
        )

asset_df = pd.DataFrame(asset_rows).sort_values(["asset_type", "relative_path"]).reset_index(drop=True)

def read_land_cover_attr(path: Path) -> str:
    with h5py.File(path, "r") as hdf:
        attr = hdf["Geometry"].attrs["Land Cover Filename"]
        return attr.decode("utf-8") if isinstance(attr, bytes) else str(attr)

land_cover_reference = read_land_cover_attr(project_folder / "Spring_BLE.g01.hdf")
compatibility_df = pd.DataFrame(
    [
        {
            "geometry_hdf": "Spring_BLE.g01.hdf",
            "land_cover_reference": land_cover_reference,
            "legacy_copy_exists": (project_folder / "Land Classification" / "LandCover (1).hdf").exists(),
            "delivery_land_cover_exists": (project_folder / "Land Cover" / "LandCover (1).hdf").exists(),
        }
    ]
)

display(asset_df)
compatibility_df


,asset_type,file,relative_path,size_mb
0,dss,Spring_BLE.dss,DSS Inputs\Spring_BLE.dss,4.806
1,land_cover_hdf,LandCover (1).hdf,Land Cover\LandCover (1).hdf,0.011
2,land_cover_tif,LandCover (1).tif,Land Cover\LandCover (1).tif,36.008
3,legacy_land_classification_hdf,LandCover (1).hdf,Land Classification\LandCover (1).hdf,0.011
4,legacy_land_classification_tif,LandCover (1).tif,Land Classification\LandCover (1).tif,36.008
5,projection,Spring_BLE_Projection.prj,Projection\Spring_BLE_Projection.prj,0.001
6,rasmap,Spring_BLE.rasmap,Spring_BLE.rasmap,0.044
7,terrain_hdf,Terrain (2).hdf,Terrain\Terrain (2).hdf,16.969
8,terrain_hdf,Terrain.hdf,Terrain\Terrain.hdf,17.161


,geometry_hdf,land_cover_reference,legacy_copy_exists,delivery_land_cover_exists
0,Spring_BLE.g01.hdf,.\Land Classification\LandCover (1).hdf,True,True


## Review Saved Audit Evidence


In [6]:
def find_latest_json(root: Path, patterns: list[str], text_patterns: list[str] | None = None) -> Path | None:
    if not root.exists():
        return None
    candidates = []
    for pattern in patterns:
        candidates.extend(root.rglob(pattern))
    candidates = sorted({path for path in candidates if path.is_file()}, key=lambda path: path.stat().st_mtime)
    if not text_patterns:
        return candidates[-1] if candidates else None
    for path in reversed(candidates):
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
        except OSError:
            continue
        if any(pattern.lower() in text.lower() for pattern in text_patterns):
            return path
    return None


def size_gb(path: Path) -> float:
    return round(path.stat().st_size / 1_000_000_000, 3)


def classify_plan_hdf(path: Path) -> dict[str, object]:
    with h5py.File(path, "r") as hdf:
        top_groups = sorted(hdf.keys())
        has_geometry = "Geometry" in hdf
        has_plan_data = "Plan Data" in hdf
        has_event_conditions = "Event Conditions" in hdf
        has_summary_only = list(hdf.keys()) == ["Results"] and "Summary" in hdf["Results"]
        if has_geometry and has_plan_data:
            classification = "hydraulic results"
        elif has_summary_only:
            classification = "compute messages only"
        else:
            classification = "unknown HDF layout"
    return {
        "file": path.name,
        "size_gb": size_gb(path),
        "classification": classification,
        "has_geometry": has_geometry,
        "has_plan_data": has_plan_data,
        "has_event_conditions": has_event_conditions,
        "top_groups": ", ".join(top_groups),
    }


def existing_path_count(values) -> int:
    return sum(bool(value) and Path(str(value)).exists() for value in values)


audit_path = find_latest_json(
    VALIDATION_ROOT,
    ["e2e_delivery_audit_*.json"],
    ["SpringRiver_11010010", "Spring River", "11010010"],
)
print(f"Audit report: {audit_path}")

audit_payload = json.loads(audit_path.read_text(encoding="utf-8")) if audit_path else []
spring_river_record = audit_payload[0] if audit_payload else {}
project_record = spring_river_record.get("projects", [{}])[0]

audit_summary = pd.DataFrame(
    [
        {
            "status": spring_river_record.get("status"),
            "hms_status": spring_river_record.get("hms_status"),
            "project_count": spring_river_record.get("project_count"),
            "audit_issues": len(spring_river_record.get("issues", [])),
            "project_crs": project_record.get("project_crs"),
            "terrain_layers": len(project_record.get("terrain_layers", [])),
            "land_cover_layers": len(project_record.get("land_cover_layers", [])),
            "dss_reference_count": project_record.get("dss_reference_count"),
            "output_hdf_count": project_record.get("output_hdf_count"),
            "outputs_inside_project": project_record.get("outputs_inside_project"),
            "crs_mismatches": len(project_record.get("terrain_crs_mismatches", []))
            + len(project_record.get("land_cover_crs_mismatches", [])),
        }
    ]
)
audit_summary


Audit report: H:\Testing\eBFE Model Organization\Validation\ebfe_delivery\e2e_delivery_audit_20260427_181627.json


,status,hms_status,project_count,audit_issues,project_crs,terrain_layers,land_cover_layers,dss_reference_count,output_hdf_count,outputs_inside_project,crs_mismatches
0,audited,not_delivered,1,0,EPSG:3433,1,1,7,7,True,0


## Review Geometry-Preprocessor Evidence


In [7]:
preprocessor_root = VALIDATION_ROOT / "preprocessor_validation"
preprocessor_path = find_latest_json(
    preprocessor_root / "spring_river_ras61_legacy_landclassification",
    ["geometry_preprocessor_validation_*.json"],
    ["Spring_BLE", "spring-river", "SpringRiver_11010010"],
)
if preprocessor_path is None:
    preprocessor_path = find_latest_json(
        preprocessor_root,
        ["geometry_preprocessor_validation_*.json"],
        ["Spring_BLE", "spring-river", "SpringRiver_11010010"],
    )
archived_preprocessor_report = preprocessor_path
print(f"Archived preprocessor report: {archived_preprocessor_report}")

preprocessor_payload = json.loads(preprocessor_path.read_text(encoding="utf-8")) if preprocessor_path else {"records": []}
records = preprocessor_payload.get("records", [])
archived_preprocessor_hdf = None
if archived_preprocessor_report is not None:
    matches = sorted(archived_preprocessor_report.parent.glob("Spring_BLE.p01.preprocessor_*.hdf"))
    archived_preprocessor_hdf = matches[-1] if matches else None

preprocessor_summary = pd.DataFrame(
    [
        {
            "generated_at": preprocessor_payload.get("generated_at"),
            "record_count": len(records),
            "status_counts": json.dumps(preprocessor_payload.get("status_counts", {}), sort_keys=True),
            "archived_preprocessor_hdf": str(archived_preprocessor_hdf) if archived_preprocessor_hdf else None,
        }
    ]
)

plan_rows = []
for record in records:
    for plan in record.get("plans", []):
        plan_rows.append(
            {
                "study": record.get("study"),
                "project": record.get("project_name"),
                "status": record.get("status"),
                "plan": plan.get("plan_number"),
                "geometry": plan.get("geometry_number"),
                "flow_type": plan.get("flow_type"),
                "elapsed_seconds": round(plan.get("elapsed_seconds", 0), 1),
                "errors": plan.get("error_count"),
                "warnings": plan.get("warning_count"),
                "return_code": plan.get("return_code"),
            }
        )

display(preprocessor_summary)
pd.DataFrame(plan_rows)


Archived preprocessor report: H:\Testing\eBFE Model Organization\Validation\ebfe_delivery\preprocessor_validation\spring_river_ras61_legacy_landclassification\geometry_preprocessor_validation_20260429_091059.json


,generated_at,record_count,status_counts,archived_preprocessor_hdf
0,20260429_091059,1,"{""passed"": 1}",H:\Testing\eBFE Model Organization\Validation\ebfe_delivery\preprocessor_validation\spring_river_ras61_legacy_landclassification\Spring_BLE.p01.preprocessor...


,study,project,status,plan,geometry,flow_type,elapsed_seconds,errors,warnings,return_code
0,spring-river,Spring_BLE,passed,01,01,Unsteady,247.3,0,0,0


## Verify Result-HDF Availability


In [8]:
organized_plan_hdfs = sorted(project_folder.glob("*.p??.hdf"))
hdf_summary = pd.DataFrame(classify_plan_hdf(path) for path in organized_plan_hdfs)
print(
    f"{existing_path_count(ras.plan_df['HDF_Results_Path'])} of {len(ras.plan_df)} plan dataframe HDF paths exist locally"
)
display(hdf_summary)

result_path_df = ras.plan_df[["plan_number", "HDF_Results_Path"]].copy()
result_path_df["exists"] = result_path_df["HDF_Results_Path"].apply(
    lambda value: bool(value) and Path(str(value)).exists()
)
result_path_df["inside_project"] = result_path_df["HDF_Results_Path"].apply(
    lambda value: bool(value)
    and Path(str(value)).exists()
    and project_folder in Path(str(value)).parents
)
result_path_df.sort_values("plan_number")


7 of 7 plan dataframe HDF paths exist locally


,file,size_gb,classification,has_geometry,has_plan_data,has_event_conditions,top_groups
0,Spring_BLE.p01.hdf,0.881,hydraulic results,True,True,True,"Event Conditions, Geometry, Plan Data, Results"
1,Spring_BLE.p02.hdf,0.878,hydraulic results,True,True,True,"Event Conditions, Geometry, Plan Data, Results"
2,Spring_BLE.p03.hdf,0.872,hydraulic results,True,True,True,"Event Conditions, Geometry, Plan Data, Results"
3,Spring_BLE.p04.hdf,0.875,hydraulic results,True,True,True,"Event Conditions, Geometry, Plan Data, Results"
4,Spring_BLE.p05.hdf,0.875,hydraulic results,True,True,True,"Event Conditions, Geometry, Plan Data, Results"
5,Spring_BLE.p06.hdf,0.887,hydraulic results,True,True,True,"Event Conditions, Geometry, Plan Data, Results"
6,Spring_BLE.p07.hdf,0.664,hydraulic results,True,True,True,"Event Conditions, Geometry, Plan Data, Results"


,plan_number,HDF_Results_Path,exists,inside_project
2,01,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p01.hdf,True,True
0,02,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p02.hdf,True,True
1,03,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p03.hdf,True,True
3,04,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p04.hdf,True,True
4,05,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p05.hdf,True,True
5,06,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p06.hdf,True,True
6,07,H:\Testing\eBFE Model Organization\Organized\SpringRiver_11010010\RAS Model\Spring_BLE.p07.hdf,True,True


## Optional Preprocessor Re-Run Command


In [ ]:
RUN_GEOMETRY_PREPROCESSOR = os.environ.get("RAS_COMMANDER_RUN_GEOMETRY_PREPROCESSOR", "0").strip().lower() not in {"0", "false", "no"}

repo_root = find_repo_root()
batch_script = repo_root / "scripts" / "ebfe_geometry_preprocessor_batch.py"
rerun_output_dir = NOTEBOOK_ARTIFACT_ROOT / "preprocessor_validation"
rerun_output_dir.mkdir(parents=True, exist_ok=True)
plan_01_hdf = project_folder / "Spring_BLE.p01.hdf"
temporary_full_result_backup = rerun_output_dir / "Spring_BLE.p01.full_results_backup.hdf"
validation_preprocessor_path = archived_preprocessor_report
validation_preprocessor_hdf = archived_preprocessor_hdf
validation_preprocessor_payload = preprocessor_payload
cmd = [
    sys.executable,
    str(batch_script),
    "--study",
    "spring-river",
    "--plan",
    "01",
    "--max-wait",
    "7200",
    "--output-dir",
    str(rerun_output_dir),
]
display_cmd = [
    "python",
    str(batch_script.relative_to(repo_root)),
    *cmd[2:],
]

if RUN_GEOMETRY_PREPROCESSOR:
    import shutil
    import subprocess

    # Pre-flight check: verify the batch script exists before attempting to run
    if not batch_script.exists():
        print(f"SKIP: Batch script not found at {batch_script}")
        print("Set RAS_COMMANDER_RUN_GEOMETRY_PREPROCESSOR=0 or ensure the script is available.")
    elif not plan_01_hdf.exists():
        raise FileNotFoundError(f"Plan 01 result HDF not found before preprocessor run: {plan_01_hdf}")
    else:
        shutil.copy2(plan_01_hdf, temporary_full_result_backup)
        print(f"Backed up full-result plan HDF: {temporary_full_result_backup}")
        print("Running Spring River geometry preprocessor validation:")
        print(" ".join(f'\"{part}\"' if " " in part else part for part in display_cmd))

        try:
            subprocess.run(cmd, check=True, timeout=7200)

            rerun_preprocessor_path = find_latest_json(
                rerun_output_dir,
                ["geometry_preprocessor_validation_*.json"],
                ["Spring_BLE", "spring-river", "SpringRiver_11010010"],
            )
            if rerun_preprocessor_path is None:
                raise FileNotFoundError(f"No rerun preprocessor report found in {rerun_output_dir}")

            validation_preprocessor_payload = json.loads(rerun_preprocessor_path.read_text(encoding="utf-8"))
            validation_records = validation_preprocessor_payload.get("records", [])
            validation_project = validation_records[0] if validation_records else {}
            validation_plan = validation_project.get("plans", [{}])[0]
            validation_generated_at = validation_preprocessor_payload.get("generated_at")
            validation_preprocessor_hdf = rerun_output_dir / f"Spring_BLE.p01.preprocessor_{validation_generated_at}.hdf"
            shutil.copy2(plan_01_hdf, validation_preprocessor_hdf)
            validation_preprocessor_path = rerun_preprocessor_path
            print(f"Fresh preprocessor report: {validation_preprocessor_path}")
            print(f"Archived fresh preprocessor HDF: {validation_preprocessor_hdf}")

            validation_summary = pd.DataFrame(
                [
                    {
                        "generated_at": validation_generated_at,
                        "status": validation_project.get("status"),
                        "plan": validation_plan.get("plan_number"),
                        "geometry": validation_plan.get("geometry_number"),
                        "elapsed_seconds": round(validation_plan.get("elapsed_seconds", 0), 1),
                        "errors": validation_plan.get("error_count"),
                        "warnings": validation_plan.get("warning_count"),
                        "preprocessor_hdf": str(validation_preprocessor_hdf),
                    }
                ]
            )
            display(validation_summary)

            if validation_plan.get("error_count") or validation_plan.get("warning_count"):
                raise AssertionError(
                    f"Fresh preprocessor run reported errors={validation_plan.get('error_count')} warnings={validation_plan.get('warning_count')}"
                )
        except subprocess.TimeoutExpired:
            print("ERROR: Geometry preprocessor timed out after 7200 seconds.")
            print("The process was killed. Check HEC-RAS installation and model configuration.")
        except subprocess.CalledProcessError as exc:
            print(f"ERROR: Geometry preprocessor failed with return code {exc.returncode}")
            print(f"Command: {exc.cmd}")
            print("Check the batch script output above for details.")
        except Exception as exc:
            print(f"ERROR: Unexpected failure during geometry preprocessor run: {exc}")
        finally:
            if temporary_full_result_backup.exists():
                shutil.copy2(temporary_full_result_backup, plan_01_hdf)
                temporary_full_result_backup.unlink()
                print(f"Restored full-result plan HDF: {plan_01_hdf}")
else:
    print("Geometry preprocessor execution disabled via RAS_COMMANDER_RUN_GEOMETRY_PREPROCESSOR.")
    print(f"Expected review artifact path: {rerun_output_dir}")
    print(" ".join(f'"{part}"' if " " in part else part for part in display_cmd))

## Write Notebook Review Artifacts


In [10]:
summary_payload = {
    "notebook": "examples/957_ebfe_spring_river_validation.ipynb",
    "organized_folder": str(organized_folder),
    "audit_report": str(audit_path) if audit_path else None,
    "archived_preprocessor_report": str(archived_preprocessor_report) if archived_preprocessor_report else None,
    "archived_preprocessor_hdf": str(archived_preprocessor_hdf) if archived_preprocessor_hdf else None,
    "validation_preprocessor_executed": RUN_GEOMETRY_PREPROCESSOR,
    "validation_preprocessor_report": str(validation_preprocessor_path) if validation_preprocessor_path else None,
    "validation_preprocessor_hdf": str(validation_preprocessor_hdf) if validation_preprocessor_hdf else None,
    "preprocessor_output_dir": str(rerun_output_dir),
    "result_hdf_count": int(len(organized_plan_hdfs)),
    "project_crs": project_record.get("project_crs"),
    "land_cover_reference": compatibility_df.iloc[0]["land_cover_reference"],
    "review_artifact_root": str(NOTEBOOK_ARTIFACT_ROOT),
}

summary_json = NOTEBOOK_ARTIFACT_ROOT / "spring_river_validation_summary.json"
summary_md = NOTEBOOK_ARTIFACT_ROOT / "spring_river_validation_summary.md"
summary_json.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")
summary_lines = [
    "# Spring River Notebook Review Summary",
    "",
    f"- Notebook: `{summary_payload['notebook']}`",
    f"- Organized folder: `{summary_payload['organized_folder']}`",
    f"- Audit report: `{summary_payload['audit_report']}`",
    f"- Archived preprocessor report: `{summary_payload['archived_preprocessor_report']}`",
    f"- Archived preprocessor HDF: `{summary_payload['archived_preprocessor_hdf']}`",
    f"- Validation preprocessor executed: `{summary_payload['validation_preprocessor_executed']}`",
    f"- Validation preprocessor report: `{summary_payload['validation_preprocessor_report']}`",
    f"- Validation preprocessor HDF: `{summary_payload['validation_preprocessor_hdf']}`",
    f"- Preprocessor output dir: `{summary_payload['preprocessor_output_dir']}`",
    f"- Result HDF count: `{summary_payload['result_hdf_count']}`",
    f"- Project CRS: `{summary_payload['project_crs']}`",
    f"- Geometry HDF land-cover reference: `{summary_payload['land_cover_reference']}`",
]
summary_md.write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Wrote review summary: {summary_json}")
print(f"Wrote review markdown: {summary_md}")


Wrote review summary: H:\Symphony\ras-commander\CLB-215\spring_river_notebook\spring_river_validation_summary.json
Wrote review markdown: H:\Symphony\ras-commander\CLB-215\spring_river_notebook\spring_river_validation_summary.md


## Delivery-Format Interpretation

Spring River is a results-ready, notebook-validated eBFE example. The
organized project remains clearly separated from Spring Creek through the
`spring-river` slug and `SpringRiver_11010010` folder naming, while still
preserving the legacy `Land Classification` compatibility copy required by
the delivered geometry HDF land-cover reference.

This validation run executes the HEC-RAS 6.1 geometry preprocessor for plan
01 / geometry 01, archives the fresh compute-message/preprocessor HDF under
`H:/Symphony/ras-commander/CLB-215/spring_river_notebook/preprocessor_validation`,
restores the delivered full-result `Spring_BLE.p01.hdf`, confirms the
organized project still resolves all seven hydraulic result HDFs, and writes
an issue-scoped review summary under
`H:/Symphony/ras-commander/CLB-215/spring_river_notebook`.
